# Run extract_tolerability_data for Semaglutide
This notebook ensures the project root is on `sys.path`, imports `extract_tolerability_data`, and runs it with `max_trials=5`.
Run the code cells below to execute the extractor.

In [2]:
import sys
import os
from pathlib import Path
# Ensure the repository root is on sys.path so imports work when running the notebook
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Added to sys.path:', repo_root)

Added to sys.path: C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai


In [7]:
# Import the extractor (adjusts to package structure in the workspace)
try:
    from medical_potential.tolerability_and_patient_burden.tolerability_tools import extract_tolerability_data, extract_incremental_trials
    from medical_potential.tolerability_and_patient_burden.tolerability_and_patient_burden import tolerability_and_patient_burden_assessment
    from medical_potential.tolerability_and_patient_burden.generate_report_and_rationale import generate_tolerability_report
    from medical_potential.gcp_utils import upload_dimension_report_pdf_to_gcs
    from medical_potential.config import TOLERABILITY_AND_PATIENT_BURDEN_DIMENSION_NAME
    print('Import succeeded: extract_tolerability_data')
except Exception as e:
    print('Import failed:', e)
    raise

Import succeeded: extract_tolerability_data


In [4]:
pdf_bytes = generate_tolerability_report(molecule='Azd-9550')

Loading latest tolerability data for 'Azd-9550' from tolerability_and_patient_burden_assessment...
  Loaded 1 row(s).
  BQ data is thin for Azd-9550 — enriching via Gemini + Google Search...


[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via tier: flex
  Enrichment complete for Azd-9550


[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 503 Service Unavailable"
[WARNING] Gemini request failed with ServerError; waiting 4.0s (retry 1/5)...
[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via tier: flex


In [9]:
upload_dimension_report_pdf_to_gcs(pdf_bytes = pdf_bytes, molecule_name = 'Azd-9550', dimension_name=TOLERABILITY_AND_PATIENT_BURDEN_DIMENSION_NAME)

[INFO] [REPORT_UPLOAD] PDF uploaded to GCS for dimension 'Tolerability & Patient Burden': gs://cognito-gcs/Cognito_new/reports_test_batch_api/Azd-9550/Medical Potential/Tolerability & Patient Burden.pdf
[INFO] [REPORT_UPLOAD] Archived PDF uploaded to GCS for dimension 'Tolerability & Patient Burden': gs://cognito-gcs/Cognito_new/reports_test_batch_api/Azd-9550/Medical Potential/archived/Tolerability & Patient Burden_20260929_101113.pdf


('gs://cognito-gcs/Cognito_new/reports_test_batch_api/Azd-9550/Medical Potential/Tolerability & Patient Burden.pdf',
 'gs://cognito-gcs/Cognito_new/reports_test_batch_api/Azd-9550/Medical Potential/archived/Tolerability & Patient Burden_20260929_101113.pdf')

In [ ]:
extract_incremental_trials(molecule_name='Tirzepatide')

In [ ]:
await tolerability_and_patient_burden_assessment('Semaglutide')

In [ ]:
result = extract_incremental_trials('Semaglutide', incremental_mode=True)

In [ ]:
len(result)

In [ ]:
# Run the extractor for Semaglutide with max_trials=5
import json

result = None
try:
    # Use top-level await in Jupyter to avoid asyncio.run() inside a running event loop
    result = await extract_tolerability_data('Semaglutide', max_trials=20)
    print('Extraction completed. Top-level keys:', list(result.keys()) if isinstance(result, dict) else type(result))
    # Print a compact JSON preview
    print(json.dumps(result, indent=2)[:2000])
except Exception as e:
    print('Extraction failed or requires external services (API keys, network):', e)
    raise

In [ ]:
import pandas as pd

pd.DataFrame(result['trials']).to_excel('tolerability_and_patient_burden_data.xlsx', index = False)